# Numerikus módszerek -- 2. gyakorlat kiegészítés
## Debuggolási feladatok: Gauss-elimináció, LU-felbontás, normák, kondíciószám

Ez a notebook **hibás implementációkat** tartalmaz a 2. gyakorlat témáihoz. Minden feladatban egy-egy Python függvény **szándékos hibát** tartalmaz.

**Feladat:** Keresd meg a hibát, javítsd ki, és ellenőrizd, hogy a javított kód helyes eredményt ad!

Minden feladathoz tartozik:
- a hibás kód,
- tesztelő kód, amely kimutatja a hibát,
- tipp a hiba megtalálásához.

**Témák:**
1. Gauss-elimináció
2. Részleges főelemkiválasztás
3. LU-felbontás
4. Előre helyettesítés (Ly = b)
5. Visszahelyettesítés (Ux = y)
6. Vektornormák
7. Mátrixnormák
8. Kondíciószám

In [ ]:
import numpy as np
from scipy.linalg import lu

np.set_printoptions(precision=4, suppress=True)

def print_augmented(Ab, step_label=""):
    """Kibővített mátrix szép kiírása."""
    n = Ab.shape[0]
    if step_label:
        print(f"--- {step_label} ---")
    for i in range(n):
        row_A = "  ".join(f"{Ab[i,j]:8.4f}" for j in range(n))
        row_b = "  ".join(f"{Ab[i,j]:8.4f}" for j in range(n, Ab.shape[1]))
        print(f"  [{row_A}  | {row_b}]")
    print()

def print_matrix(M, label=""):
    """Mátrix szép kiírása."""
    if label:
        print(label)
    for i in range(M.shape[0]):
        row = "  ".join(f"{M[i,j]:8.4f}" for j in range(M.shape[1]))
        print(f"  [{row}]")
    print()

---
## Debug 1: Hibás Gauss-elimináció

Az alábbi Gauss-elimináció implementáció **nem adja vissza a helyes felső háromszög alakot**. A függvény lefut, de az eredmény nem felső háromszögmátrix.

*Tipp:* Vizsgáld meg, hogy az eliminációs lépésben pontosan mely elemeket módosítjuk! Az `Ab[i, ...]` sor frissítésénél hol kellene kezdődnie a tartománynak?

In [ ]:
def gauss_elim_hibas(A, b):
    """HIBÁS Gauss-elimináció. Keresd meg a hibát!"""
    n = len(b)
    Ab = np.column_stack([A.astype(float), b.astype(float)])
    
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k+1:] = Ab[i, k+1:] - m * Ab[k, k+1:]  # <-- VIZSGÁLD MEG!
    
    # Visszahelyettesítés
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, n] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

# Tesztelés
A1 = np.array([[3, -1, 2],
               [1,  2, -1],
               [2, -3,  5]], dtype=float)
b1 = np.array([7, 0, 11], dtype=float)

x_hibas = gauss_elim_hibas(A1, b1)
x_helyes = np.linalg.solve(A1, b1)

print(f"Hibás megoldás:  x = {x_hibas}")
print(f"Helyes megoldás: x = {x_helyes}")
print(f"Egyeznek? {np.allclose(x_hibas, x_helyes)}")
print(f"\nEllenőrzés: A @ x_hibás = {A1 @ x_hibas}")
print(f"b = {b1}")
print(f"A @ x_hibás = b? {np.allclose(A1 @ x_hibas, b1)}")
print("\n>>> A hiba az eliminációs lépésben van.")
print(">>> Tipp: mi történik az Ab[i, k] elemmel (amit nullázni kellene)?")

---
## Debug 2: Hibás részleges főelemkiválasztás

Az alábbi GE implementáció részleges főelemkiválasztással dolgozik, de **hibásan keresi a főelemet**. Egyes mátrixoknál rossz eredményt ad.

*Tipp:* A főelemet a $k$-adik lépésben melyik tartományban kell keresni? Az egész oszlopban vagy csak a $k$-adik sortól lefelé?

In [ ]:
def gauss_pivot_hibas(A, b):
    """HIBÁS GE részleges főelemkiválasztással. Keresd meg a hibát!"""
    n = len(b)
    Ab = np.column_stack([A.astype(float), b.astype(float)])
    
    for k in range(n - 1):
        # Főelemkiválasztás
        max_idx = np.argmax(np.abs(Ab[0:n, k])) # <-- VIZSGÁLD MEG!
        
        if Ab[max_idx, k] == 0:
            raise ValueError("Szinguláris mátrix!")
        
        if max_idx != k:
            Ab[[k, max_idx]] = Ab[[max_idx, k]]  # sorcsere
        
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
    
    # Visszahelyettesítés
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, n] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

# Tesztelés: ez a mátrix kimutatja a hibát
A2 = np.array([[2, -1,  3],
               [4,  2, -1],
               [1, -3,  2]], dtype=float)
b2 = np.array([5, 3, 0], dtype=float)

x_hibas = gauss_pivot_hibas(A2, b2)
x_helyes = np.linalg.solve(A2, b2)

print(f"Hibás megoldás:  x = {x_hibas}")
print(f"Helyes megoldás: x = {x_helyes}")
print(f"Egyeznek? {np.allclose(x_hibas, x_helyes)}")
print(f"\nEllenőrzés: A @ x_hibás = {A2 @ x_hibas}")
print(f"b = {b2}")
print(f"A @ x_hibás = b? {np.allclose(A2 @ x_hibas, b2)}")
print("\n>>> A hiba a főelemkiválasztás tartományában van.")
print(">>> Tipp: a 2. lépésben a már feldolgozott sorokat is megvizsgálja?")

---
## Debug 3: Hibás LU-felbontás

Az alábbi LU-felbontás GE-vel **hibásan tárolja az $L$ mátrix elemeit**. A GE lépések helyesek, de $L \cdot U \neq A$.

*Tipp:* Az $L$ mátrixba a GE-s hányadosok ($m_{ik}$) kerülnek. Milyen előjellel?

In [ ]:
def lu_hibas(A):
    """HIBÁS LU-felbontás. Keresd meg a hibát!"""
    n = A.shape[0]
    U = A.astype(float).copy()
    L = np.eye(n)
    
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]
            L[i, k] = -m   # <-- VIZSGÁLD MEG!
            U[i, k:] -= m * U[k, k:]
    
    return L, U

# Tesztelés
A3 = np.array([[2, 0, 3],
               [-4, 5, -2],
               [6, -5, 4]], dtype=float)

L_h, U_h = lu_hibas(A3)

print_matrix(L_h, "L (hibás) =")
print_matrix(U_h, "U =")
print_matrix(L_h @ U_h, "L @ U =")
print_matrix(A3, "A =")
print(f"L @ U = A? {np.allclose(L_h @ U_h, A3)}")
print("\n>>> A GE lépések helyesek (U jó), de L @ U ≠ A.")
print(">>> Tipp: az L_k mátrixban -m_ik van, de az L = L_1^{-1}·L_2^{-1}·... mátrixban mi?")

---
## Debug 4: Hibás előre helyettesítés ($Ly = b$)

Az alábbi előre helyettesítés **rossz irányban** halad. Az $Ly = b$ rendszert az $L$ alsó háromszög struktúrája miatt felülről lefelé kellene megoldani.

*Tipp:* Milyen irányban kell haladni az előre helyettesítésnél? Felülről lefelé, vagy alulról felfelé?

In [ ]:
def forward_sub_hibas(L, b):
    """HIBÁS előre helyettesítés (Ly = b). Keresd meg a hibát!"""
    n = len(b)
    y = np.zeros(n)
    for i in range(n - 1, -1, -1):   # <-- VIZSGÁLD MEG!
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i, i]
    return y

def backward_sub(U, y):
    """Helyes visszahelyettesítés (Ux = y)."""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x

# Tesztelés
A4 = np.array([[4, -2, 1],
               [-2, 4, -2],
               [1, -2, 4]], dtype=float)
b4 = np.array([5, -8, 13], dtype=float)

# Helyes LU-felbontás
from scipy.linalg import lu as scipy_lu
P, L4, U4 = scipy_lu(A4)
assert np.allclose(P, np.eye(3)), "P = I kell legyen"

# Hibás megoldás
y_hibas = forward_sub_hibas(L4, b4)
x_hibas = backward_sub(U4, y_hibas)

x_helyes = np.linalg.solve(A4, b4)

print(f"y (hibás előre helyettesítés): {y_hibas}")
print(f"x (hibás):  {x_hibas}")
print(f"x (helyes): {x_helyes}")
print(f"Egyeznek? {np.allclose(x_hibas, x_helyes)}")
print(f"\nA @ x_hibás = {A4 @ x_hibas}")
print(f"b = {b4}")
print("\n>>> A hiba a ciklus irányában van.")
print(">>> Tipp: az Ly = b alsó háromszögű rendszert felülről lefelé oldjuk meg,")
print(">>> mert y_1 = b_1/l_11, majd y_2 = (b_2 - l_21*y_1)/l_22, stb.")

---
## Debug 5: Hibás visszahelyettesítés ($Ux = y$)

Az alábbi visszahelyettesítés implementáció **rossz indexeléssel** számol. A belső szorzatba bevonja az átlóelemet is.

*Tipp:* A visszahelyettesítés képlete:
$$x_i = \frac{y_i - \sum_{j=i+1}^{n} u_{ij} x_j}{u_{ii}}$$
A szumma $j = i+1$-től indul, nem $j = i$-től!

In [ ]:
def backward_sub_hibas(U, y):
    """HIBÁS visszahelyettesítés (Ux = y). Keresd meg a hibát!"""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i:], x[i:])) / U[i, i]   # <-- VIZSGÁLD MEG!
    return x

# Tesztelés
U5 = np.array([[2, 0, 3],
               [0, 5, 4],
               [0, 0, -1]], dtype=float)
y5 = np.array([-1, 1, 1], dtype=float)

x_hibas = backward_sub_hibas(U5, y5)
x_helyes = np.linalg.solve(U5, y5)

print(f"x (hibás):  {x_hibas}")
print(f"x (helyes): {x_helyes}")
print(f"Egyeznek? {np.allclose(x_hibas, x_helyes)}")
print(f"\nU @ x_hibás = {U5 @ x_hibas}")
print(f"y = {y5}")
print(f"U @ x_hibás = y? {np.allclose(U5 @ x_hibas, y5)}")
print("\n>>> A hiba: U[i, i:] tartalmazza az u_ii átlóelemet is,")
print(">>> de a szummázás csak j = i+1-től kellene induljon.")
print(">>> Mi lesz az x_i képletben, ha az u_ii * x_i is belekerül a szorzatba?")

---
## Debug 6: Hibás vektornorma-számítás

Az alábbi kézzel írt norma-számító függvény **két hibát** is tartalmaz. Az egyiknél hiányzik az abszolút érték, a másiknál rossz a szélsőérték-keresés iránya.

*Tipp 1:* Az 1-norma definíciója: $\|x\|_1 = \sum_i |x_i|$. Mi történik, ha elhagyjuk az abszolút értéket negatív elemek esetén?

*Tipp 2:* Az $\infty$-norma a maximumot keresi, nem a minimumot!

In [ ]:
def norma_1_hibas(x):
    """HIBÁS 1-norma. Keresd meg a hibát!"""
    return np.sum(x)        # <-- VIZSGÁLD MEG!

def norma_2(x):
    """Helyes 2-norma."""
    return np.sqrt(np.sum(x**2))

def norma_inf_hibas(x):
    """HIBÁS inf-norma. Keresd meg a hibát!"""
    return np.min(np.abs(x))  # <-- VIZSGÁLD MEG!

# Tesztelés
v = np.array([-3, 4, 0, -1], dtype=float)

print(f"v = {v}")
print(f"\n{'Norma':>12} {'Hibás':>10} {'Helyes (numpy)':>15} {'Egyezik?':>10}")
print("-" * 50)

n1_h = norma_1_hibas(v)
n1_ok = np.linalg.norm(v, 1)
print(f"{'||v||_1':>12} {n1_h:>10.4f} {n1_ok:>15.4f} {str(np.isclose(n1_h, n1_ok)):>10}")

n2_h = norma_2(v)
n2_ok = np.linalg.norm(v, 2)
print(f"{'||v||_2':>12} {n2_h:>10.4f} {n2_ok:>15.4f} {str(np.isclose(n2_h, n2_ok)):>10}")

ninf_h = norma_inf_hibas(v)
ninf_ok = np.linalg.norm(v, np.inf)
print(f"{'||v||_inf':>12} {ninf_h:>10.4f} {ninf_ok:>15.4f} {str(np.isclose(ninf_h, ninf_ok)):>10}")

# Egyenlőtlenségek ellenőrzése a hibás értékekkel
print(f"\nEgyenlőtlenség a hibás normákkal:")
print(f"  ||v||_inf ≤ ||v||_2 ≤ ||v||_1 ?")
print(f"  {ninf_h:.4f} ≤ {n2_h:.4f} ≤ {n1_h:.4f} ? {ninf_h <= n2_h and n2_h <= n1_h}")
print(f"  → Hibás normákkal az egyenlőtlenség SÉRÜL!")
print("\n>>> Két hiba van: az 1-normánál és az inf-normánál.")

---
## Debug 7: Hibás mátrixnorma-számítás

Az alábbi kézzel írt mátrixnorma-számító függvények **felcserélik** az 1-normát és az $\infty$-normát.

*Emlékeztető:*
- $\|A\|_1 = \max_j \sum_i |a_{ij}|$ — maximális **oszlopösszeg**
- $\|A\|_\infty = \max_i \sum_j |a_{ij}|$ — maximális **sorösszeg**

*Tipp:* Melyik a sor és melyik az oszlop irány? `axis=0` az oszlopok mentén összegez (→ soronkénti összeg), `axis=1` a sorok mentén.

In [ ]:
def matrix_norm_1_hibas(A):
    """HIBÁS 1-norma (oszlopnorma). Keresd meg a hibát!"""
    return np.max(np.sum(np.abs(A), axis=1))   # <-- VIZSGÁLD MEG!

def matrix_norm_inf_hibas(A):
    """HIBÁS inf-norma (sornorma). Keresd meg a hibát!"""
    return np.max(np.sum(np.abs(A), axis=0))   # <-- VIZSGÁLD MEG!

# Tesztelés: nem szimmetrikus mátrix (szimmetrikusnál nem derülne ki!)
A7 = np.array([[1, -4, 0],
               [2,  2, -1],
               [0, -1,  3]], dtype=float)

print_matrix(A7, "A =")

# Oszlopösszegek: |1|+|2|+|0|=3, |-4|+|2|+|-1|=7, |0|+|-1|+|3|=4
# Sorösszegek:    |1|+|-4|+|0|=5, |2|+|2|+|-1|=5, |0|+|-1|+|3|=4
print("Oszlopösszegek (abs):")
for j in range(3):
    tagok = ' + '.join(f'|{A7[i,j]:.0f}|' for i in range(3))
    print(f"  {j+1}. oszlop: {tagok} = {np.sum(np.abs(A7[:,j])):.0f}")

print("Sorösszegek (abs):")
for i in range(3):
    tagok = ' + '.join(f'|{A7[i,j]:.0f}|' for j in range(3))
    print(f"  {i+1}. sor: {tagok} = {np.sum(np.abs(A7[i,:])):.0f}")

print(f"\n{'Norma':>12} {'Hibás':>10} {'Helyes (numpy)':>15} {'Egyezik?':>10}")
print("-" * 50)

n1_h = matrix_norm_1_hibas(A7)
n1_ok = np.linalg.norm(A7, 1)
print(f"{'||A||_1':>12} {n1_h:>10.4f} {n1_ok:>15.4f} {str(np.isclose(n1_h, n1_ok)):>10}")

ninf_h = matrix_norm_inf_hibas(A7)
ninf_ok = np.linalg.norm(A7, np.inf)
print(f"{'||A||_inf':>12} {ninf_h:>10.4f} {ninf_ok:>15.4f} {str(np.isclose(ninf_h, ninf_ok)):>10}")

print(f"\n>>> Helyes értékek: ||A||_1 = {n1_ok:.0f} (max oszlopösszeg), ||A||_inf = {ninf_ok:.0f} (max sorösszeg)")
print(">>> A hibás kód felcseréli a kettőt! axis=0 és axis=1 meg van keverve.")

---
## Debug 8: Hibás kondíciószám-számítás

Az alábbi kód a mátrix kondíciószámát próbálja kiszámítani, de a **képlet hibás**.

*Emlékeztető:* A kondíciószám definíciója:
$$\kappa(A) = \|A\| \cdot \|A^{-1}\|$$

*Tipp:* A szorzást könnyű összekeverni az osztással! A perturbációs korlát:
$$\frac{\|\Delta x\|}{\|x\|} \le \kappa(A) \cdot \frac{\|\Delta b\|}{\|b\|}$$

Ha a kiszámolt kondíciószám túl kicsi, a korlát nem fog teljesülni.

In [ ]:
def kondicio_hibas(A, p=2):
    """HIBÁS kondíciószám-számítás. Keresd meg a hibát!"""
    norm_A = np.linalg.norm(A, p)
    norm_Ainv = np.linalg.norm(np.linalg.inv(A), p)
    return norm_A / norm_Ainv   # <-- VIZSGÁLD MEG!

# Tesztelés: rosszul kondicionált mátrix
A8 = np.array([[1, 2],
               [1.001, 2]], dtype=float)
b8 = np.array([3, 3.001], dtype=float)

kappa_hibas = kondicio_hibas(A8, 2)
kappa_helyes = np.linalg.cond(A8, 2)

print(f"κ (hibás):  {kappa_hibas:.4f}")
print(f"κ (helyes): {kappa_helyes:.4f}")
print(f"Egyeznek? {np.isclose(kappa_hibas, kappa_helyes)}")

# Perturbációs kísérlet
x_exact = np.linalg.solve(A8, b8)
delta_b = np.array([1e-4, 0])
x_pert = np.linalg.solve(A8, b8 + delta_b)

rel_hiba_b = np.linalg.norm(delta_b) / np.linalg.norm(b8)
rel_hiba_x = np.linalg.norm(x_pert - x_exact) / np.linalg.norm(x_exact)

print(f"\nPerturbációs kísérlet:")
print(f"  ||Δb||/||b|| = {rel_hiba_b:.6e}")
print(f"  ||Δx||/||x|| = {rel_hiba_x:.6e}")
print(f"\n  Hibás korlát: κ_hibás · rel_hiba_b = {kappa_hibas * rel_hiba_b:.6e}")
print(f"  ||Δx||/||x|| ≤ korlát? {rel_hiba_x <= kappa_hibas * rel_hiba_b + 1e-15}")
print(f"  → A hibás kondíciószámmal a korlát NEM teljesül!")
print(f"\n  Helyes korlát: κ · rel_hiba_b = {kappa_helyes * rel_hiba_b:.6e}")
print(f"  ||Δx||/||x|| ≤ korlát? {rel_hiba_x <= kappa_helyes * rel_hiba_b + 1e-15}")
print(f"  → A helyes kondíciószámmal a korlát teljesül!")
print("\n>>> A hiba: osztás van szorzás helyett a kondíciószám képletében!")

---
## Összefoglaló: tipikus hibák

| # | Téma | Hiba típusa | Tanulság |
|---|------|-------------|----------|
| 1 | GE eliminációs lépés | `k+1:` helyett `k:` — az eliminált elem nem nullázódik | Mindig ellenőrizd, hogy a felső háromszög valóban felső háromszög! |
| 2 | Főelemkiválasztás | `0:n` helyett `k:n` — már feldolgozott sorok is beleszámítanak | A főelemet csak a $k$-adik sortól lefelé keressük! |
| 3 | LU-felbontás | $-m$ van $m$ helyett az $L$-ben | $L_k$-ban $-m_{ik}$, de $L = L_1^{-1} \cdots$-ben $+m_{ik}$! |
| 4 | Előre helyettesítés | Alulról felfelé halad felülről lefelé helyett | $Ly = b$: felülről lefelé, $Ux = y$: alulról felfelé! |
| 5 | Visszahelyettesítés | `U[i, i:]` tartalmazza az átlóelemet is | A szumma $j = i+1$-től indul, nem $i$-től! |
| 6 | Vektornormák | Hiányzó abs(), min vs max | Definíciók pontos betartása: $\sum|x_i|$, $\max|x_i|$ |
| 7 | Mátrixnormák | axis=0 és axis=1 felcserélve | 1-norma: oszlopösszeg (axis=0), $\infty$: sorösszeg (axis=1) |
| 8 | Kondíciószám | Osztás szorzás helyett | $\kappa(A) = \|A\| \cdot \|A^{-1}\|$, nem $\|A\| / \|A^{-1}\|$ |